# Фінальний проєкт: аналіз датасету infectious_cases у PostgreSQL

Mykyta Oliinyk

Статистика інфекційних захворювань за країнами, регіонами та роками. Порядок роботи: імпорт CSV у схему `pandemic`, нормалізація до 3NF, агрегати `Number_rabies`, колонка різниці в роках, дві власні SQL-функції. Усі оператори також зібрані у файлі `final_project.sql`.

## Підготовка середовища

PostgreSQL запускається через `pgserver` у каталозі `/tmp/fp_pg`. Notebook розрахований на запуск у Google Colab і локально на Python 3.12.

In [1]:
%%capture
%pip install -q pgserver sqlalchemy psycopg2-binary pandas "psycopg[binary]"

In [2]:
import pandas as pd
import pgserver
from pathlib import Path
from sqlalchemy import create_engine, text
from sqlalchemy.exc import IntegrityError
from urllib.request import urlopen

pg = pgserver.get_server('/tmp/fp_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
with engine.connect() as probe:
    print(probe.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


`SET search_path` діє лише в межах сесії, тому всі запити нижче йдуть через одне відкрите з'єднання `conn`. Інше з'єднання з пулу мало б `search_path` за замовчуванням і не знаходило б таблиць схеми `pandemic` без префікса.

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

conn = engine.connect()


def run(sql):
    """Виконує DDL/DML в одній транзакції."""
    conn.execute(text(sql))
    conn.commit()


def show(sql):
    """Виконує SELECT і повертає результат як таблицю."""
    result = pd.read_sql_query(text(sql), conn, coerce_float=False)
    conn.commit()
    return result.astype(object).where(result.notna(), None)

## Завдання 1. Завантаження даних

### 1.1. Схема pandemic як схема за замовчуванням

In [4]:
run("""
CREATE SCHEMA IF NOT EXISTS pandemic;
SET search_path TO pandemic;
""")

In [5]:
show("""
SELECT current_schema() AS default_schema,
       current_setting('search_path') AS search_path,
       current_setting('server_version') AS server_version;
""")

,default_schema,search_path,server_version
0,pandemic,pandemic,16.2


### 1.2. Імпорт CSV у таблицю infectious_cases

Файл береться з робочого каталогу. Якщо його немає, notebook завантажує його напряму з Google Drive за id файла.

In [6]:
FILE_ID = '1dQf-uVXzYPYYGgXsG1y4Q1qDdhP2miZl'
URL = f'https://drive.usercontent.google.com/download?id={FILE_ID}&export=download&confirm=t'
CSV_PATH = Path('infectious_cases.csv')
HEADER = 'Entity,Code,Year,'

if not CSV_PATH.exists():
    with urlopen(URL, timeout=120) as response:
        payload = response.read()
    if not payload.startswith(HEADER.encode()):
        raise ValueError('Google Drive повернув не CSV')
    CSV_PATH.write_bytes(payload)

print(CSV_PATH.name, CSV_PATH.stat().st_size, 'байт')

infectious_cases.csv 540895 байт


Сира таблиця зберігає CSV без змін: усі колонки мають тип `TEXT`, порожні значення лишаються порожніми рядками. Типи й `NULL` з'являються на етапі нормалізації, де кожне перетворення можна перевірити.

Імпорт виконано так само, як у домашніх роботах курсу: `pandas.read_csv` і `to_sql`. `keep_default_na=False` не дає pandas перетворити порожні комірки на `NaN`, а `dtype=str` зберігає значення як є.

In [7]:
run("""
DROP TABLE IF EXISTS disease_stats;
DROP TABLE IF EXISTS entities;
DROP TABLE IF EXISTS infectious_cases;

CREATE TABLE infectious_cases (
    entity               TEXT,
    code                 TEXT,
    year                 TEXT,
    number_yaws          TEXT,
    polio_cases          TEXT,
    cases_guinea_worm    TEXT,
    number_rabies        TEXT,
    number_malaria       TEXT,
    number_hiv           TEXT,
    number_tuberculosis  TEXT,
    number_smallpox      TEXT,
    number_cholera_cases TEXT
);
""")

In [8]:
raw_df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
raw_df.columns = raw_df.columns.str.lower()

raw_df.to_sql('infectious_cases', conn, schema='pandemic', if_exists='append',
              index=False, chunksize=2_000)
conn.commit()
print('Рядків у CSV:', len(raw_df))

Рядків у CSV: 10521


### 1.3. Перегляд даних

In [9]:
show("""
SELECT *
FROM infectious_cases
LIMIT 5;
""")

,entity,code,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,Afghanistan,AFG,1980,,6160,0,,,,,,
1,Afghanistan,AFG,1981,,5859,0,,,,,,
2,Afghanistan,AFG,1982,,9730,0,,,,,,
3,Afghanistan,AFG,1983,,13937,0,,,,,,
4,Afghanistan,AFG,1984,,3864,0,,,,,,


In [10]:
show("""
SELECT ordinal_position, column_name, data_type
FROM information_schema.columns
WHERE table_schema = 'pandemic'
  AND table_name = 'infectious_cases'
ORDER BY ordinal_position;
""")

,ordinal_position,column_name,data_type
0,1,entity,text
1,2,code,text
2,3,year,text
3,4,number_yaws,text
4,5,polio_cases,text
5,6,cases_guinea_worm,text
6,7,number_rabies,text
7,8,number_malaria,text
8,9,number_hiv,text
9,10,number_tuberculosis,text


In [11]:
show("""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT (entity, code)) AS entity_code_pairs,
       COUNT(*) FILTER (WHERE code = '') AS blank_code,
       COUNT(*) FILTER (WHERE number_rabies = '') AS blank_number_rabies,
       MIN(year) AS min_year,
       MAX(year) AS max_year
FROM infectious_cases;
""")

,total_rows,entity_code_pairs,blank_code,blank_number_rabies,min_year,max_year
0,10521,245,1128,3741,1920,2023


In [12]:
show("""
SELECT COUNT(DISTINCT entity) AS entities_without_code,
       string_agg(DISTINCT entity, ', ') AS entity_list
FROM infectious_cases
WHERE code = '';
""")

,entities_without_code,entity_list
0,34,"Africa, African Region (WHO), Americas, Asia, East Asia & Pacific (WB), Eastern Mediterranean, Eastern Mediterranean Region (WHO), England, Europe, Europe & Central Asia (WB), European Region (WHO), G20, High Income (WB), Latin America & Caribbean (WB), Low Income (WB), Lower Middle Income (WB), Middle East & North Africa (WB), Middle Income (WB), North America, North America (WB), Northern Ireland, OECD Countries, Oceania, Region of the Americas (WHO), Scotland, Serbia and Montenegro (former), South America, South Asia (WB), South-East Asia, South-East Asia Region (WHO), Sub-Saharan Africa (WB), Wales, Western Pacific, Western Pacific Region (WHO)"


In [13]:
show(r"""
SELECT t.column_name,
       COUNT(*) FILTER (WHERE t.value = '') AS blank,
       COUNT(*) FILTER (WHERE t.value ~ '^[0-9]+(\.[0-9]+)?$') AS numeric_values,
       COUNT(*) FILTER (WHERE t.value <> '' AND t.value !~ '^[0-9]+(\.[0-9]+)?$') AS other_values
FROM infectious_cases
CROSS JOIN LATERAL (
    VALUES
        ('year', year),
        ('number_yaws', number_yaws),
        ('polio_cases', polio_cases),
        ('cases_guinea_worm', cases_guinea_worm),
        ('number_rabies', number_rabies),
        ('number_malaria', number_malaria),
        ('number_hiv', number_hiv),
        ('number_tuberculosis', number_tuberculosis),
        ('number_smallpox', number_smallpox),
        ('number_cholera_cases', number_cholera_cases)
) AS t (column_name, value)
GROUP BY t.column_name
ORDER BY t.column_name;
""")

,column_name,blank,numeric_values,other_values
0,cases_guinea_worm,2093,8428,0
1,number_cholera_cases,7554,2967,0
2,number_hiv,3741,6780,0
3,number_malaria,3741,6780,0
4,number_rabies,3741,6780,0
5,number_smallpox,10417,104,0
6,number_tuberculosis,3741,6780,0
7,number_yaws,10447,74,0
8,polio_cases,2911,7610,0
9,year,0,10521,0


Що видно з результатів:

- у таблиці 10521 рядок і 245 унікальних пар `(entity, code)`, роки від 1920 до 2023;
- `code` порожній у 1128 рядках 34 сутностей. Це регіони й групи країн (`Africa`, `G20`, `OECD Countries`), а також, наприклад, `England`, `Scotland`, `Wales`, `Northern Ireland`;
- `number_rabies` порожній у 3741 рядку, значення є у 6780 рядках;
- у `year` і колонках показників є лише порожні рядки та числа (`other_values` = 0), тому `NULLIF(value, '')::NUMERIC` виконується без помилок.

## Завдання 2. Нормалізація до 3NF

### 2.1. Функціональні залежності

Перед розбиттям таблиці перевіряємо на даних залежності, на яких воно ґрунтується. У кожному рядку результату `violations` має дорівнювати 0.

In [14]:
show("""
SELECT 'entity -> code' AS dependency, COUNT(*) AS violations
FROM (
    SELECT entity
    FROM infectious_cases
    GROUP BY entity
    HAVING COUNT(DISTINCT code) > 1
) AS v
UNION ALL
SELECT 'code -> entity', COUNT(*)
FROM (
    SELECT code
    FROM infectious_cases
    WHERE code <> ''
    GROUP BY code
    HAVING COUNT(DISTINCT entity) > 1
) AS v
UNION ALL
SELECT '(entity, year) -> row', COUNT(*)
FROM (
    SELECT entity, year
    FROM infectious_cases
    GROUP BY entity, year
    HAVING COUNT(*) > 1
) AS v;
""")

,dependency,violations
0,entity -> code,0
1,code -> entity,0
2,"(entity, year) -> row",0


### 2.2. Обґрунтування 3NF

Вихідна таблиця: `infectious_cases(entity, code, year, 9 показників)`. Перевірка вище дала 0 порушень для всіх трьох залежностей. `entity -> code` рахує порожній код як окреме значення, тому сутність, яка мала б код лише в частині рядків, теж дала б порушення. `code -> entity` перевіряється лише для заповнених кодів: порожній код означає його відсутність, а не спільний код. Кандидатний ключ: `(entity, year)`. Пара `(code, year)` ключем бути не може, бо в 34 сутностей коду немає.

- **`infectious_cases`: 1NF є, 2NF порушена.** Значення атомарні, дублікатів за ключем немає. Дев'ять колонок показників не є повторюваною групою: кожна має власний зміст (сказ, малярія, ВІЛ тощо), а не перелік однотипних значень. Але `code` залежить лише від `entity`, тобто від частини ключа, тому пара назва-код повторюється в кожному році. Це винесено в таблицю `entities` із сурогатним ключем `entity_id`.
- **`entities`.** 1NF: у рядку одна назва й не більше одного коду. 2NF: ключ `entity_id` складається з однієї колонки, тому часткових залежностей бути не може. 3NF: `entity -> code` і `code -> entity` не є транзитивними залежностями, бо і `entity`, і заповнений `code` визначають увесь рядок: перший є кандидатним ключем, другий унікальний серед наявних кодів. Обмеження `UNIQUE (entity)` і `UNIQUE (code)` закріплюють це в схемі.
- **`disease_stats`.** 1NF: значення атомарні, рядок визначається ключем `(entity_id, year)`. 2NF: показник описує факт "захворювання X у сутності E в році Y", тому залежить від усього ключа, а не лише від `entity_id` чи `year`. 3NF: `entity` і `code` сюди не входять, а показники не визначають один одного, тож транзитивних залежностей немає.

**Чому показники лишаються разом у другій таблиці.** Умова вимагає рівно дві таблиці. Широка форма вже відповідає 3NF, бо всі показники мають однаковий ключ. Довга форма `(entity_id, year, disease, value)` не додала б нормальності: назва захворювання повторювалася б текстом або потребувала б третьої таблиці-довідника. Окрема таблиця `year` теж нічого не дає, бо рік не має власних атрибутів.

Рішення щодо типів і порожніх значень:

- порожній `code` зберігається як `NULL`. `UNIQUE (code)` вважає `NULL` різними значеннями, тому допускає 34 сутності без коду й водночас забороняє двом сутностям мати один код. Дублікат `('Africa', NULL)` блокує `UNIQUE (entity)`, з якого випливає й унікальність пари `(entity, code)`;
- показники мають тип `NUMERIC`, бо частина значень дробова, а `NUMERIC` зберігає їх точно. Порожнє значення стає `NULL`: "немає даних", а не нуль.

### 2.3. Створення нормалізованих таблиць

In [15]:
run("""
DROP TABLE IF EXISTS disease_stats;
DROP TABLE IF EXISTS entities;

CREATE TABLE entities (
    entity_id INTEGER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    entity    TEXT NOT NULL,
    code      TEXT,
    CONSTRAINT uq_entities_entity UNIQUE (entity),
    CONSTRAINT uq_entities_code UNIQUE (code)
);

CREATE TABLE disease_stats (
    entity_id            INTEGER NOT NULL REFERENCES entities (entity_id),
    year                 INTEGER NOT NULL,
    number_yaws          NUMERIC,
    polio_cases          NUMERIC,
    cases_guinea_worm    NUMERIC,
    number_rabies        NUMERIC,
    number_malaria       NUMERIC,
    number_hiv           NUMERIC,
    number_tuberculosis  NUMERIC,
    number_smallpox      NUMERIC,
    number_cholera_cases NUMERIC,
    PRIMARY KEY (entity_id, year)
);
""")

In [16]:
show("""
SELECT table_name, ordinal_position, column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_schema = 'pandemic'
  AND table_name IN ('entities', 'disease_stats')
ORDER BY table_name DESC, ordinal_position;
""")

,table_name,ordinal_position,column_name,data_type,is_nullable
0,entities,1,entity_id,integer,NO
1,entities,2,entity,text,NO
2,entities,3,code,text,YES
3,disease_stats,1,entity_id,integer,NO
4,disease_stats,2,year,integer,NO
5,disease_stats,3,number_yaws,numeric,YES
6,disease_stats,4,polio_cases,numeric,YES
7,disease_stats,5,cases_guinea_worm,numeric,YES
8,disease_stats,6,number_rabies,numeric,YES
9,disease_stats,7,number_malaria,numeric,YES


In [17]:
show("""
SELECT conrelid::regclass AS table_name,
       conname AS constraint_name,
       pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid IN ('entities'::regclass, 'disease_stats'::regclass)
  AND contype IN ('p', 'u', 'f')
ORDER BY table_name DESC, contype, conname;
""")

,table_name,constraint_name,definition
0,disease_stats,disease_stats_entity_id_fkey,FOREIGN KEY (entity_id) REFERENCES entities(entity_id)
1,disease_stats,disease_stats_pkey,"PRIMARY KEY (entity_id, year)"
2,entities,entities_pkey,PRIMARY KEY (entity_id)
3,entities,uq_entities_code,UNIQUE (code)
4,entities,uq_entities_entity,UNIQUE (entity)


### 2.4. Заповнення таблиць

`id` підставляється через з'єднання сирої таблиці з `entities`. З'єднання йде за парою `(entity, code)`. Для `code` використано `IS NOT DISTINCT FROM`, бо `NULL = NULL` не дає `TRUE`, і звичайна рівність загубила б усі рядки сутностей без коду.

Після заповнення дві клітинки показують перші рядки обох таблиць: у `disease_stats` замість `entity` і `code` стоїть `entity_id`.

In [18]:
run("""
INSERT INTO entities (entity, code)
SELECT DISTINCT entity, NULLIF(code, '')
FROM infectious_cases
ORDER BY entity, NULLIF(code, '');

INSERT INTO disease_stats (
    entity_id,
    year,
    number_yaws,
    polio_cases,
    cases_guinea_worm,
    number_rabies,
    number_malaria,
    number_hiv,
    number_tuberculosis,
    number_smallpox,
    number_cholera_cases
)
SELECT
    e.entity_id,
    r.year::INTEGER,
    NULLIF(r.number_yaws, '')::NUMERIC,
    NULLIF(r.polio_cases, '')::NUMERIC,
    NULLIF(r.cases_guinea_worm, '')::NUMERIC,
    NULLIF(r.number_rabies, '')::NUMERIC,
    NULLIF(r.number_malaria, '')::NUMERIC,
    NULLIF(r.number_hiv, '')::NUMERIC,
    NULLIF(r.number_tuberculosis, '')::NUMERIC,
    NULLIF(r.number_smallpox, '')::NUMERIC,
    NULLIF(r.number_cholera_cases, '')::NUMERIC
FROM infectious_cases AS r
JOIN entities AS e
  ON e.entity = r.entity
 AND e.code IS NOT DISTINCT FROM NULLIF(r.code, '');
""")

In [19]:
show("""
SELECT *
FROM entities
ORDER BY entity_id
LIMIT 5;
""")

,entity_id,entity,code
0,1,Afghanistan,AFG
1,2,Africa,None
2,3,African Region (WHO),None
3,4,Albania,ALB
4,5,Algeria,DZA


In [20]:
show("""
SELECT *
FROM disease_stats
WHERE number_rabies IS NOT NULL
ORDER BY entity_id, year
LIMIT 5;
""")

,entity_id,year,number_yaws,polio_cases,cases_guinea_worm,number_rabies,number_malaria,number_hiv,number_tuberculosis,number_smallpox,number_cholera_cases
0,1,1990,None,336,0,2.145772,1219845.8,87.665596,16303.112,None,None
1,1,1991,None,14,0,2.3209748,1132781.5,103.77359,17582.254,None,None
2,1,1992,None,None,0,2.6180408,1081609.6,119.9334,19786.672,None,None
3,1,1993,None,None,0,2.8531172,963817.1,127.2633,21195.934,None,37046
4,1,1994,None,None,0,2.9899871,818779.5,133.06308,21658.172,None,38735


In [21]:
show("""
SELECT 'infectious_cases: rows' AS check_name, COUNT(*) AS value
FROM infectious_cases
UNION ALL
SELECT 'disease_stats: rows', COUNT(*)
FROM disease_stats
UNION ALL
SELECT 'distinct (entity, code) in raw', COUNT(DISTINCT (entity, code))
FROM infectious_cases
UNION ALL
SELECT 'entities: rows', COUNT(*)
FROM entities
UNION ALL
SELECT 'entities: code IS NULL', COUNT(*)
FROM entities
WHERE code IS NULL
UNION ALL
SELECT 'disease_stats: rows of entities without code', COUNT(*)
FROM disease_stats AS s
JOIN entities AS e USING (entity_id)
WHERE e.code IS NULL
UNION ALL
SELECT 'rows if joined with plain e.code = r.code', COUNT(*)
FROM infectious_cases AS r
JOIN entities AS e
  ON e.entity = r.entity
 AND e.code = NULLIF(r.code, '');
""")

,check_name,value
0,infectious_cases: rows,10521
1,disease_stats: rows,10521
2,"distinct (entity, code) in raw",245
3,entities: rows,245
4,entities: code IS NULL,34
5,disease_stats: rows of entities without code,1128
6,rows if joined with plain e.code = r.code,9393


### 2.5. Перевірка імпорту

Кількість рядків збігається: 10521 у `infectious_cases` і `disease_stats`, 245 пар у сирій таблиці й в `entities`. Рядки сутностей без коду на місці: 34 сутності, 1128 рядків. Рядок `rows if joined with plain e.code = r.code` показує 9393: саме стільки лишилося б після з'єднання через звичайну рівність, тобто 1128 рядків було б втрачено.

Збіг кількості не доводить збігу значень, тому нижче вихідна таблиця відновлюється зі з'єднання двох нормалізованих і порівнюється із сирою через `EXCEPT ALL` в обидва боки. Друга клітинка перевіряє, що обмеження `entities` справді блокують порушення залежностей: дублікат із порожнім кодом, другий код для сутності та код, який уже належить іншій сутності.

In [22]:
show("""
WITH typed_raw AS (
    SELECT entity,
           NULLIF(code, '') AS code,
           year::INTEGER AS year,
           NULLIF(number_yaws, '')::NUMERIC AS number_yaws,
           NULLIF(polio_cases, '')::NUMERIC AS polio_cases,
           NULLIF(cases_guinea_worm, '')::NUMERIC AS cases_guinea_worm,
           NULLIF(number_rabies, '')::NUMERIC AS number_rabies,
           NULLIF(number_malaria, '')::NUMERIC AS number_malaria,
           NULLIF(number_hiv, '')::NUMERIC AS number_hiv,
           NULLIF(number_tuberculosis, '')::NUMERIC AS number_tuberculosis,
           NULLIF(number_smallpox, '')::NUMERIC AS number_smallpox,
           NULLIF(number_cholera_cases, '')::NUMERIC AS number_cholera_cases
    FROM infectious_cases
),
restored AS (
    SELECT e.entity, e.code, s.year,
           number_yaws, polio_cases, cases_guinea_worm,
           number_rabies, number_malaria, number_hiv,
           number_tuberculosis, number_smallpox, number_cholera_cases
    FROM disease_stats AS s
    JOIN entities AS e USING (entity_id)
)
SELECT
    (SELECT COUNT(*) FROM (SELECT * FROM typed_raw EXCEPT ALL SELECT * FROM restored) AS d) AS only_in_raw,
    (SELECT COUNT(*) FROM (SELECT * FROM restored EXCEPT ALL SELECT * FROM typed_raw) AS d) AS only_in_normalized;
""")

,only_in_raw,only_in_normalized
0,0,0


In [23]:
for values in ("('Africa', NULL)", "('Africa', 'AFR')", "('New entity', 'AFG')"):
    try:
        run(f'INSERT INTO entities (entity, code) VALUES {values}')
        print(values, 'прийнято')
    except IntegrityError as error:
        conn.rollback()
        print(values, 'відхилено:', error.orig.diag.constraint_name)

('Africa', NULL) відхилено: uq_entities_entity
('Africa', 'AFR') відхилено: uq_entities_entity
('New entity', 'AFG') відхилено: uq_entities_code


Розбіжностей немає в обидва боки (0 і 0), тож декомпозиція без втрат. Усі три вставки відхилено: перші дві обмеженням `uq_entities_entity`, третю обмеженням `uq_entities_code`.

## Завдання 3. Агрегати Number_rabies

### 3.1. Порожні значення

Порожній `Number_rabies` означає відсутність даних, а не нуль, тому заміняти його на 0 не можна: це занизило б середнє. `AVG`, `MIN`, `MAX` і `SUM` пропускають `NULL` самі.

Є інша проблема: якщо в сутності немає жодного значення, `AVG` дає `NULL`, а `ORDER BY ... DESC` у PostgreSQL за замовчуванням ставить `NULL` першими. Без фільтра такі сутності займають верхні рядки, що й видно в першій клітинці. Друга рахує їх кількість.

In [24]:
show("""
SELECT e.entity_id, e.entity, e.code,
       COUNT(s.number_rabies) AS years_with_value,
       AVG(s.number_rabies) AS avg_rabies
FROM entities AS e
JOIN disease_stats AS s USING (entity_id)
GROUP BY e.entity_id, e.entity, e.code
ORDER BY avg_rabies DESC, e.entity_id
LIMIT 3;
""")

,entity_id,entity,code,years_with_value,avg_rabies
0,2,Africa,None,0,None
1,7,Americas,None,0,None
2,13,Asia,None,0,None


In [25]:
show("""
SELECT COUNT(*) AS entities_total,
       COUNT(*) FILTER (WHERE years_with_value = 0) AS entities_without_values,
       COUNT(*) FILTER (WHERE years_with_value > 0) AS entities_with_values
FROM (
    SELECT s.entity_id, COUNT(s.number_rabies) AS years_with_value
    FROM disease_stats AS s
    GROUP BY s.entity_id
) AS per_entity;
""")

,entities_total,entities_without_values,entities_with_values
0,245,19,226


### 3.2. Топ-10 за середнім значенням

Сутностей без жодного значення 19 із 245, значення є у 226. Рядки без значення відсікає `WHERE s.number_rabies IS NOT NULL`, тому в рейтинг потрапляють лише сутності з даними. Сортування йде за неокругленим `AVG`, `entity_id` лише фіксує порядок за рівних значень. `years_with_value` показує, зі скількох років пораховано агрегати.

In [26]:
show("""
SELECT e.entity_id,
       e.entity,
       e.code,
       COUNT(s.number_rabies) AS years_with_value,
       ROUND(AVG(s.number_rabies), 2) AS avg_rabies,
       ROUND(MIN(s.number_rabies), 2) AS min_rabies,
       ROUND(MAX(s.number_rabies), 2) AS max_rabies,
       ROUND(SUM(s.number_rabies), 2) AS sum_rabies
FROM entities AS e
JOIN disease_stats AS s USING (entity_id)
WHERE s.number_rabies IS NOT NULL
GROUP BY e.entity_id, e.entity, e.code
ORDER BY AVG(s.number_rabies) DESC, e.entity_id
LIMIT 10;
""")

,entity_id,entity,code,years_with_value,avg_rabies,min_rabies,max_rabies,sum_rabies
0,241,World,OWID_WRL,30,20192.37,14075.51,24744.66,605771.11
1,121,Lower Middle Income (WB),None,30,15193.96,10202.53,19182.80,455818.78
2,202,South Asia (WB),None,30,11729.89,7271.28,15361.88,351896.66
3,206,South-East Asia Region (WHO),None,30,11424.33,6806.01,15641.96,342729.81
4,77,G20,None,30,10189.05,6339.08,13164.88,305671.38
5,97,India,IND,30,8599.17,5425.87,11121.14,257975.20
6,209,Sub-Saharan Africa (WB),None,30,4748.27,4356.82,5017.98,142448.04
7,3,African Region (WHO),None,30,4649.70,4283.07,4917.78,139491.07
8,120,Low Income (WB),None,30,3568.11,3014.62,3873.53,107043.16
9,58,East Asia & Pacific (WB),None,30,3427.46,2009.94,4591.01,102823.83


У всіх десяти рядків по 30 значень, тому середні можна порівнювати між собою. Найбільше середнє у `World`: 20192.37 на рік, мінімум 14075.51, максимум 24744.66, сума 605771.11.

Лише один рядок із десяти є окремою країною: `India` із середнім 8599.17. Решта рядків: світ, регіони та групи країн, які перетинаються між собою, тому порівнювати їх із країнами або підсумовувати не варто. Відповіддю на завдання є таблиця вище, бо умова вимагає рахувати для кожної унікальної пари.

### 3.3. Додатковий зріз: лише країни

Запит той самий, але без сутностей із порожнім `code` і кодами `OWID_*`. Перша клітинка показує, що цими кодами позначені світ і держави, яких уже немає. Такий фільтр спрощений: серед сутностей без коду є не лише регіони, а й, наприклад, `England` чи `Wales`.

In [27]:
show("""
SELECT entity, code
FROM entities
WHERE code LIKE 'OWID%'
ORDER BY code;
""")

,entity,code
0,Serbia and Montenegro,OWID_SRM
1,World,OWID_WRL
2,Yugoslavia,OWID_YGS


In [28]:
show("""
SELECT e.entity_id,
       e.entity,
       e.code,
       COUNT(s.number_rabies) AS years_with_value,
       ROUND(AVG(s.number_rabies), 2) AS avg_rabies,
       ROUND(MIN(s.number_rabies), 2) AS min_rabies,
       ROUND(MAX(s.number_rabies), 2) AS max_rabies,
       ROUND(SUM(s.number_rabies), 2) AS sum_rabies
FROM entities AS e
JOIN disease_stats AS s USING (entity_id)
WHERE s.number_rabies IS NOT NULL
  AND e.code IS NOT NULL
  AND e.code NOT LIKE 'OWID%'
GROUP BY e.entity_id, e.entity, e.code
ORDER BY AVG(s.number_rabies) DESC, e.entity_id
LIMIT 10;
""")

,entity_id,entity,code,years_with_value,avg_rabies,min_rabies,max_rabies,sum_rabies
0,97,India,IND,30,8599.17,5425.87,11121.14,257975.20
1,163,Pakistan,PAK,30,1582.17,1177.14,1881.73,47465.09
2,151,Nigeria,NGA,30,1335.42,1073.95,1441.12,40062.46
3,42,China,CHN,30,1247.91,744.25,2075.59,37437.20
4,70,Ethiopia,ETH,30,1145.17,758.05,1323.40,34355.18
5,143,Myanmar,MMR,30,972.90,560.70,1449.83,29187.11
6,19,Bangladesh,BGD,30,785.71,180.40,1734.71,23571.26
7,146,Nepal,NPL,30,661.25,338.14,1041.63,19837.53
8,170,Philippines,PHL,30,618.69,358.06,898.07,18560.73
9,98,Indonesia,IDN,30,267.71,124.62,481.37,8031.44


Серед країн за `India` йдуть `Pakistan` (1582.17) і `Nigeria` (1335.42). Це абсолютні кількості, тому без даних про населення вони не показують рівень захворюваності.

## Завдання 4. Колонка різниці в роках

### 4.1. Запит із вбудованими функціями

- `make_date(year, 1, 1)` будує дату 1 січня відповідного року;
- `CURRENT_DATE` дає поточну дату під час виконання запиту;
- `age(CURRENT_DATE, ...)` рахує інтервал між двома датами, а `EXTRACT(YEAR FROM ...)` бере з нього кількість повних років.

Запит охоплює всі рядки: `LIMIT 10` стоїть лише заради перегляду. Значення не зберігаються в таблиці, бо `CURRENT_DATE` щодня інша, а згенерована колонка потребує незмінного виразу. Дати у виводі відповідають дню запуску notebook.

In [29]:
show("""
SELECT e.entity,
       s.year,
       make_date(s.year, 1, 1) AS year_start,
       CURRENT_DATE AS today,
       EXTRACT(YEAR FROM age(CURRENT_DATE, make_date(s.year, 1, 1)))::INTEGER AS years_diff
FROM disease_stats AS s
JOIN entities AS e USING (entity_id)
ORDER BY e.entity, s.year
LIMIT 10;
""")

,entity,year,year_start,today,years_diff
0,Afghanistan,1960,1960-01-01,2026-11-04,66
1,Afghanistan,1965,1965-01-01,2026-11-04,61
2,Afghanistan,1980,1980-01-01,2026-11-04,46
3,Afghanistan,1981,1981-01-01,2026-11-04,45
4,Afghanistan,1982,1982-01-01,2026-11-04,44
5,Afghanistan,1983,1983-01-01,2026-11-04,43
6,Afghanistan,1984,1984-01-01,2026-11-04,42
7,Afghanistan,1985,1985-01-01,2026-11-04,41
8,Afghanistan,1986,1986-01-01,2026-11-04,40
9,Afghanistan,1987,1987-01-01,2026-11-04,39


### 4.2. Перевірка на всій таблиці

`mismatches` порівнює результат `age()` зі звичайним відніманням років. Оскільки початкова дата завжди 1 січня, значення мають збігатися для кожного рядка.

In [30]:
show("""
WITH diff AS (
    SELECT s.year,
           EXTRACT(YEAR FROM age(CURRENT_DATE, make_date(s.year, 1, 1)))::INTEGER AS years_diff
    FROM disease_stats AS s
)
SELECT COUNT(*) AS total_rows,
       COUNT(years_diff) AS rows_with_diff,
       MIN(year) AS min_year,
       MAX(year) AS max_year,
       MIN(years_diff) AS min_diff,
       MAX(years_diff) AS max_diff,
       COUNT(*) FILTER (
           WHERE years_diff <> EXTRACT(YEAR FROM CURRENT_DATE)::INTEGER - year
       ) AS mismatches
FROM diff;
""")

,total_rows,rows_with_diff,min_year,max_year,min_diff,max_diff,mismatches
0,10521,10521,1920,2023,3,106,0


Різниця порахована для кожного з 10521 рядка, `mismatches` = 0.

## Завдання 5. Власні функції

### 5.1. Функція years_since

Приймає рік і повертає різницю в роках між поточною датою та 1 січня цього року, тобто будує той самий атрибут, що й у завданні 4. Функція позначена `STABLE`, бо залежить від `CURRENT_DATE`: з `IMMUTABLE` PostgreSQL міг би вважати результат незмінним у будь-який день.

Клітинки нижче показують приклад з умови для 1996 року, використання на таблиці та збіг із виразом із завдання 4 на всіх рядках.

In [31]:
run("""
CREATE OR REPLACE FUNCTION years_since(p_year INTEGER)
RETURNS INTEGER
LANGUAGE sql
STABLE
AS $$
    SELECT EXTRACT(YEAR FROM age(CURRENT_DATE, make_date(p_year, 1, 1)))::INTEGER;
$$;
""")

In [32]:
show("""
SELECT 1996 AS year_value,
       make_date(1996, 1, 1) AS year_start,
       CURRENT_DATE AS today,
       years_since(1996) AS years_diff;
""")

,year_value,year_start,today,years_diff
0,1996,1996-01-01,2026-11-04,30


In [33]:
show("""
SELECT e.entity,
       s.year,
       years_since(s.year) AS years_diff
FROM disease_stats AS s
JOIN entities AS e USING (entity_id)
ORDER BY e.entity, s.year
LIMIT 10;
""")

,entity,year,years_diff
0,Afghanistan,1960,66
1,Afghanistan,1965,61
2,Afghanistan,1980,46
3,Afghanistan,1981,45
4,Afghanistan,1982,44
5,Afghanistan,1983,43
6,Afghanistan,1984,42
7,Afghanistan,1985,41
8,Afghanistan,1986,40
9,Afghanistan,1987,39


In [34]:
show("""
SELECT COUNT(*) AS total_rows,
       COUNT(*) FILTER (
           WHERE years_since(s.year) IS DISTINCT FROM
                 EXTRACT(YEAR FROM age(CURRENT_DATE, make_date(s.year, 1, 1)))::INTEGER
       ) AS mismatches_with_task_4
FROM disease_stats AS s;
""")

,total_rows,mismatches_with_task_4
0,10521,0


### 5.2. Функція avg_cases_per_period

Приймає кількість захворювань на рік і довільний дільник, повертає середню кількість за період. `NULLIF(p_divisor, 0)` повертає `NULL` для нульового дільника замість помилки `division by zero`, а `NULL` у кількості дає `NULL` у результаті. Функція `IMMUTABLE`, бо результат визначається лише аргументами.

Перевірка на простих значеннях: 1200 на рік дає 100 на місяць, 300 на квартал і 600 на півріччя. Останні дві колонки показують крайні випадки.

In [35]:
run("""
CREATE OR REPLACE FUNCTION avg_cases_per_period(p_cases_per_year NUMERIC, p_divisor NUMERIC)
RETURNS NUMERIC
LANGUAGE sql
IMMUTABLE
AS $$
    SELECT p_cases_per_year / NULLIF(p_divisor, 0);
$$;
""")

In [36]:
show("""
SELECT p.proname AS function_name,
       pg_get_function_arguments(p.oid) AS arguments,
       pg_get_function_result(p.oid) AS returns,
       CASE p.provolatile WHEN 'i' THEN 'IMMUTABLE' WHEN 's' THEN 'STABLE' ELSE 'VOLATILE' END AS volatility
FROM pg_proc AS p
JOIN pg_namespace AS n ON n.oid = p.pronamespace
WHERE n.nspname = 'pandemic'
ORDER BY p.proname;
""")

,function_name,arguments,returns,volatility
0,avg_cases_per_period,"p_cases_per_year numeric, p_divisor numeric",numeric,IMMUTABLE
1,years_since,p_year integer,integer,STABLE


In [37]:
show("""
SELECT ROUND(avg_cases_per_period(1200, 12), 2) AS per_month,
       ROUND(avg_cases_per_period(1200, 4), 2) AS per_quarter,
       ROUND(avg_cases_per_period(1200, 2), 2) AS per_half_year,
       avg_cases_per_period(NULL, 12) AS null_cases,
       avg_cases_per_period(1200, 0) AS zero_divisor;
""")

,per_month,per_quarter,per_half_year,null_cases,zero_divisor
0,100.00,300.00,600.00,None,None


### 5.3. Використання обох функцій у запиті

Дільники: 12 для місяця, 4 для кварталу, 2 для півріччя. У запит беруться лише рядки з числовим значенням. У `disease_stats` колонка має тип `NUMERIC`, порожні значення там `NULL`, тому досить умови `IS NOT NULL`. Показник береться той самий, що й у завданні 3, `number_rabies`. `LIMIT 10` лише для перегляду.

In [38]:
show("""
SELECT e.entity,
       s.year,
       years_since(s.year) AS years_diff,
       ROUND(s.number_rabies, 4) AS cases_per_year,
       ROUND(avg_cases_per_period(s.number_rabies, 12), 4) AS avg_per_month,
       ROUND(avg_cases_per_period(s.number_rabies, 4), 4) AS avg_per_quarter,
       ROUND(avg_cases_per_period(s.number_rabies, 2), 4) AS avg_per_half_year
FROM disease_stats AS s
JOIN entities AS e USING (entity_id)
WHERE s.number_rabies IS NOT NULL
ORDER BY e.entity, s.year
LIMIT 10;
""")

,entity,year,years_diff,cases_per_year,avg_per_month,avg_per_quarter,avg_per_half_year
0,Afghanistan,1990,36,2.1458,0.1788,0.5364,1.0729
1,Afghanistan,1991,35,2.3210,0.1934,0.5802,1.1605
2,Afghanistan,1992,34,2.6180,0.2182,0.6545,1.3090
3,Afghanistan,1993,33,2.8531,0.2378,0.7133,1.4266
4,Afghanistan,1994,32,2.9900,0.2492,0.7475,1.4950
5,Afghanistan,1995,31,3.1264,0.2605,0.7816,1.5632
6,Afghanistan,1996,30,3.2814,0.2735,0.8204,1.6407
7,Afghanistan,1997,29,3.4461,0.2872,0.8615,1.7231
8,Afghanistan,1998,28,3.5570,0.2964,0.8893,1.7785
9,Afghanistan,1999,27,3.6070,0.3006,0.9018,1.8035


### 5.4. Перевірка на всіх числових рядках

Рахуємо те саме двома шляхами: з `disease_stats` через `IS NOT NULL` і із сирої таблиці, де значення текстові, тому фільтр робить регулярний вираз, а `::NUMERIC` застосовується вже до відібраних рядків. Множення результату назад на дільник має дати вихідну суму.

In [39]:
show(r"""
SELECT 'disease_stats' AS source,
       COUNT(*) AS rows_used,
       SUM(s.number_rabies) AS sum_cases,
       SUM(avg_cases_per_period(s.number_rabies, 12) * 12)::NUMERIC(20, 6) AS sum_month_x12,
       SUM(avg_cases_per_period(s.number_rabies, 4) * 4)::NUMERIC(20, 6) AS sum_quarter_x4,
       SUM(avg_cases_per_period(s.number_rabies, 2) * 2)::NUMERIC(20, 6) AS sum_half_year_x2
FROM disease_stats AS s
WHERE s.number_rabies IS NOT NULL
UNION ALL
SELECT 'infectious_cases (raw)',
       COUNT(*),
       SUM(r.number_rabies::NUMERIC),
       SUM(avg_cases_per_period(r.number_rabies::NUMERIC, 12) * 12)::NUMERIC(20, 6),
       SUM(avg_cases_per_period(r.number_rabies::NUMERIC, 4) * 4)::NUMERIC(20, 6),
       SUM(avg_cases_per_period(r.number_rabies::NUMERIC, 2) * 2)::NUMERIC(20, 6)
FROM infectious_cases AS r
WHERE r.number_rabies ~ '^[0-9]+(\.[0-9]+)?$';
""")

,source,rows_used,sum_cases,sum_month_x12,sum_quarter_x4,sum_half_year_x2
0,disease_stats,6780,3334448.921412666900,3334448.921413,3334448.921413,3334448.921413
1,infectious_cases (raw),6780,3334448.921412666900,3334448.921413,3334448.921413,3334448.921413


Обидва шляхи відбирають однакові 6780 рядків і дають однакову суму 3334448.92. Решта 3741 рядок без значення у функцію не потрапляє. Множення назад на 12, 4 і 2 дає ту саму суму в межах шести знаків після коми.

## Підсумок

| Завдання | Результат |
|---|---|
| 1. Імпорт | схема `pandemic`, таблиця `infectious_cases`: 10521 рядок |
| 2. Нормалізація | `entities`: 245 рядків, `disease_stats`: 10521 рядок, розбіжностей із сирою таблицею 0 |
| 3. Агрегати | топ-10 за `AVG(number_rabies)`, перший рядок `World` (20192.37), 19 сутностей без значень не враховано |
| 4. Різниця в роках | `make_date`, `CURRENT_DATE`, `age`, `EXTRACT`, порахована для всіх рядків |
| 5. Функції | `years_since`, `avg_cases_per_period` із дільниками 12, 4, 2 на 6780 числових рядках |